# Laboratorium: segmentacja klientów i predykcja odejść

Dane: Telco Customer Churn (IBM), 7043 klientów operatora telekomunikacyjnego. Dane są prawdziwe, bez żadnych zmian.
Pytanie firmy: kto odejdzie w przyszłym miesiącu i co z tym zrobić.

Ściąga ze składnią pandas i scikit-learn jest na samym końcu notatnika. Najszybciej trafisz do niej przez spis treści po lewej stronie Colaba.

Jak zacząć:

1. Plik > Zapisz kopię na Dysku. Pracujesz na swojej kopii.
2. Uruchom komórkę 1 w sekcji Start (Shift+Enter).
3. Jeśli pojawi się błąd, podnieś rękę.

W każdym zadaniu uzupełniasz linie oznaczone `# TODO`. Ostatnia komórka zadania to samokontrola: jeśli wypisze OK, wynik jest poprawny.

## Start

Komórka 1 importuje biblioteki i wczytuje dane z publicznego adresu. Powinien pojawić się rozmiar (7043, 21) i pięć pierwszych wierszy.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    silhouette_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

# Wykresy czytelne na rzutniku: duży rozmiar i czcionka 14 pt
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
})

# Kolory laboratorium; segmenty dostają je w tej kolejności
GOLD, SEA, PLUM, GRAPHITE, GREY = "#D4A017", "#2A9D8F", "#7B4B94", "#23272F", "#9AA0A6"
SEGMENT_COLORS = [GOLD, SEA, PLUM, GRAPHITE, GREY]
SEA_CMAP = LinearSegmentedColormap.from_list("morski", ["#FFFFFF", SEA])

RANDOM_STATE = 42

DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(DATA_URL)
print("Rozmiar danych:", df.shape)
df.head()

In [ ]:
# Komórka zapasowa: uruchom ją tylko wtedy, gdy komórka 1 nie pobrała danych z adresu URL.
# Usuń znaki # z początku linii poniżej (Ctrl+/), uruchom i wybierz plik Telco-Customer-Churn.csv.
# try:
#     from google.colab import files
#     uploaded = files.upload()
#     df = pd.read_csv(next(iter(uploaded)))
# except ImportError:
#     # Lokalny Jupyter bez Colaba: plik leży w katalogu data/ repozytorium
#     df = pd.read_csv("../data/Telco-Customer-Churn.csv")
# print("Rozmiar danych:", df.shape)

In [ ]:
# Samokontrola: Start
assert "df" in globals(), "Nie ma tabeli df. Uruchom komórkę 1 albo komórkę zapasową."
assert df.shape == (7043, 21), "Dane wczytały się niekompletnie. Uruchom komórkę 1 jeszcze raz albo użyj komórki zapasowej."
print("Start: OK")

## Zadanie 1: Eksploracja (12 min)

Uzupełnij luki `# TODO`:

- odsetek odejść w całym zbiorze, jako ułamek od 0 do 1, w zmiennej `churn_rate`,
- odsetek odejść według rodzaju umowy `Contract` w zmiennej `churn_by_contract` (wykres słupkowy jest gotowy),
- odsetek odejść według jednej kolumny kategorycznej, którą wybierzesz sam, w zmiennej `churn_by_choice`.

Histogram stażu `tenure` w podziale na Churn jest gotowy. Uruchom go i zobacz, gdzie leżą odchodzący.

Pytanie kontrolne: który sposób płatności (`PaymentMethod`) ma największy odsetek odejść i ile on wynosi?

In [ ]:
# Odsetek klientów, którzy odeszli (Churn == "Yes")
# TODO: porównaj kolumnę Churn z wartością "Yes" i weź średnią z wyniku.
print(f"Odsetek odejść w całym zbiorze: {churn_rate:.1%}")

In [ ]:
# Odsetek odejść w każdym rodzaju umowy
# TODO: zgrupuj porównanie Churn == "Yes" według df["Contract"] i policz średnią.
print((churn_by_contract * 100).round(1))

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(churn_by_contract.index, churn_by_contract.values * 100, color=SEA)
ax.set_title("Odsetek odejść według rodzaju umowy")
ax.set_xlabel("Rodzaj umowy")
ax.set_ylabel("Odsetek odejść [%]")
plt.show()

In [ ]:
# Rozkład stażu osobno dla zostających i odchodzących
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(data=df, x="tenure", hue="Churn", bins=36, palette={"No": SEA, "Yes": GOLD}, alpha=0.7, ax=ax)
ax.set_title("Staż klienta a odejścia")
ax.set_xlabel("Staż [miesiące]")
ax.set_ylabel("Liczba klientów")
ax.get_legend().set_title("Odszedł (Churn)")
plt.show()

In [ ]:
# Kolumna do wyboru: odsetek odejść w każdej jej kategorii
# TODO: wpisz nazwę wybranej kolumny kategorycznej, np. "PaymentMethod", "InternetService" albo "PaperlessBilling".
# TODO: policz odsetek odejść według chosen_column tak samo jak dla umowy i posortuj malejąco.
print((churn_by_choice * 100).round(1))

### Twoja obserwacja

Co widać w wybranej kolumnie? Wpisz jedno lub dwa zdania.

Odpowiedź: ...

In [ ]:
# Samokontrola: Zadanie 1
assert "churn_rate" in globals(), "Brakuje zmiennej churn_rate. Uzupełnij pierwszą lukę."
assert 0 < churn_rate < 1, "churn_rate ma być ułamkiem od 0 do 1, a nie liczbą klientów ani procentem."
assert abs(churn_rate - 0.265) < 0.001, "Odsetek odejść nie zgadza się z danymi. Sprawdź, czy liczysz klientów z Churn równym \"Yes\"."
assert "churn_by_contract" in globals(), "Brakuje zmiennej churn_by_contract. Uzupełnij lukę z umową."
assert set(churn_by_contract.index) == set(df["Contract"].unique()), "churn_by_contract ma mieć jedną wartość na każdy rodzaj umowy."
assert churn_by_contract.between(0, 1).all(), "Odsetki według umowy mają być ułamkami od 0 do 1."
assert "churn_by_choice" in globals(), "Brakuje zmiennej churn_by_choice. Uzupełnij lukę z wybraną kolumną."
assert set(churn_by_choice.index) == set(df[chosen_column].unique()), "churn_by_choice ma mieć jedną wartość na każdą kategorię wybranej kolumny."
print("Zadanie 1: OK")

## Zadanie 2: Przygotowanie danych (13 min)

Uzupełnij luki `# TODO`:

- zamień `TotalCharges` z tekstu na liczbę funkcją `pd.to_numeric` z `errors="coerce"`, puste pola staną się brakami (NaN),
- wybierz wiersze z brakiem w `TotalCharges` i zobacz ich `tenure`. Kim są ci klienci?
- usuń te wiersze, to klienci, którzy jeszcze nie zapłacili pierwszego rachunku,
- usuń kolumnę `customerID`,
- zamień `Churn` na liczby: 1 = odszedł, 0 = został,
- zamień kolumny kategoryczne na kolumny 0/1 funkcją `pd.get_dummies(..., drop_first=True)`,
- przeskaluj `tenure`, `MonthlyCharges` i `TotalCharges` przez `StandardScaler` do osobnej macierzy `X_cluster`, która posłuży do grupowania.

Drzewo decyzyjne skalowania nie potrzebuje: dzieli dane progami na jednej cesze naraz, więc skala cechy nie ma znaczenia. Dlatego `df_model` zostaje bez skalowania.

Pytanie kontrolne: ile wierszy zostaje po czyszczeniu i jaki staż mają usunięci klienci?

In [ ]:
# TotalCharges jest zapisane jako tekst; puste pola zamieniamy na NaN
df_clean = df.copy()
# TODO: zamień df_clean["TotalCharges"] na liczby funkcją pd.to_numeric, tak żeby puste pola dały NaN.
# TODO: wybierz z df_clean wiersze, w których TotalCharges jest brakiem (isna).
print("Wiersze z brakiem w TotalCharges:", len(missing_rows))
missing_rows[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

In [ ]:
# Usuwamy nowych klientów bez pierwszego rachunku (tenure == 0)
# TODO: usuń wiersze z brakiem w TotalCharges (dropna z parametrem subset).
# Identyfikator klienta nie niesie informacji dla modelu
# TODO: usuń kolumnę customerID.
# Churn jako liczba: 1 = odszedł, 0 = został
# TODO: zamień wartości "Yes" i "No" w kolumnie Churn na 1 i 0 metodą map.
df_clean = df_clean.reset_index(drop=True)
print("Rozmiar po czyszczeniu:", df_clean.shape)

In [ ]:
# Kolumny tekstowe zamieniamy na kolumny 0/1; drop_first usuwa po jednej kategorii z każdej kolumny
categorical_cols = df_clean.select_dtypes(exclude="number").columns.tolist()
print("Kolumny kategoryczne:", len(categorical_cols))
# TODO: zakoduj categorical_cols funkcją pd.get_dummies z drop_first=True i dtype=int.
print("Rozmiar tabeli dla modelu:", df_model.shape)
df_model.head()

In [ ]:
# Do grupowania bierzemy trzy cechy liczbowe i sprowadzamy je do jednej skali
cluster_features = ["tenure", "MonthlyCharges", "TotalCharges"]
scaler = StandardScaler()
# TODO: dopasuj scaler do df_clean[cluster_features] i przekształć te kolumny (fit_transform).
print("Średnie po skalowaniu:", X_cluster.mean(axis=0).round(3))
print("Odchylenia po skalowaniu:", X_cluster.std(axis=0).round(3))

In [ ]:
# Samokontrola: Zadanie 2
assert "df_clean" in globals(), "Brakuje tabeli df_clean."
assert pd.api.types.is_numeric_dtype(df_clean["TotalCharges"]), "TotalCharges nadal nie jest liczbą. Sprawdź pierwszą lukę."
assert len(df_clean) == 7032, "Liczba wierszy po czyszczeniu się nie zgadza. Usuń tylko wiersze z brakiem w TotalCharges."
assert df_clean.isna().sum().sum() == 0, "W df_clean zostały braki danych."
assert "customerID" not in df_clean.columns, "Kolumna customerID nadal jest w df_clean."
assert set(df_clean["Churn"].unique()) == {0, 1}, "Churn ma zawierać tylko liczby 0 i 1."
assert "df_model" in globals() and df_model.select_dtypes(exclude="number").empty, "W df_model zostały kolumny tekstowe. Zakoduj wszystkie kolumny kategoryczne."
assert "X_cluster" in globals() and X_cluster.shape == (7032, 3), "X_cluster ma mieć 7032 wiersze i 3 kolumny."
assert np.allclose(X_cluster.mean(axis=0), 0, atol=1e-6), "Kolumny X_cluster mają mieć średnią 0. Sprawdź skalowanie."
print("Zadanie 2: OK")

## Zadanie 3: Segmentacja K-means (15 min)

Grupujemy klientów tylko po trzech przeskalowanych cechach z `X_cluster`. Kolumna `Churn` nie bierze w tym udziału.

Uzupełnij luki `# TODO`:

- pętla po k od 2 do 8: dla każdego k model `KMeans(n_clusters=k, random_state=42, n_init=10)`, inercja (`inertia_`) i silhouette (`silhouette_score`), wyniki w listach `inertias` i `silhouettes`,
- wybór k w zmiennej `chosen_k`: kieruj się silhouette, ale segmenty muszą dać się nazwać,
- model końcowy `kmeans_model` i kolumna `segment` w `df_clean`,
- tabela profilu: średnie trzech cech i liczba klientów w każdym segmencie,
- odsetek odejść w każdym segmencie. Dopiero teraz patrzymy na Churn.

Na końcu nadaj segmentom nazwy po polsku.

Pytanie kontrolne: ile razy różni się odsetek odejść między najbezpieczniejszym a najbardziej ryzykownym segmentem?

In [ ]:
k_values = list(range(2, 9))
inertias = []
silhouettes = []
# TODO: w pętli po k_values dopasuj KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10) do X_cluster i dopisz do list inercję oraz silhouette.
pd.DataFrame({"k": k_values, "inercja": inertias, "silhouette": silhouettes}).round(3)

In [ ]:
# Metoda łokcia i silhouette obok siebie
fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(16, 6))
ax_left.plot(k_values, inertias, marker="o", color=GRAPHITE, linewidth=2)
ax_left.set_title("Inercja dla kolejnych k")
ax_left.set_xlabel("Liczba segmentów k")
ax_left.set_ylabel("Inercja")
ax_right.plot(k_values, silhouettes, marker="o", color=SEA, linewidth=2)
ax_right.set_title("Silhouette dla kolejnych k")
ax_right.set_xlabel("Liczba segmentów k")
ax_right.set_ylabel("Silhouette")
plt.tight_layout()
plt.show()

### Wybór k

Najwyższe silhouette ma k = 2 (0,480), potem k = 4 (0,472) i k = 3 (0,452). Różnice są małe.

Przy k = 2 drugi segment liczy 4672 klientów i miesza dwie różne grupy: nowych klientów z drogim pakietem i klientów z najtańszym pakietem. Takiego segmentu nie da się nazwać jednym profilem, a odsetki odejść w obu segmentach to tylko 16,9 % i 31,5 %.

Dlatego bierzemy k = 3. Każdy segment ma ponad 2000 klientów i własny profil, który łatwo opisać firmie. Jeśli wybierzesz k = 4, to też da się obronić, samokontrola sprawdza wybrane przez Ciebie k.

In [ ]:
# TODO: wpisz wybrane k na podstawie silhouette i tego, czy segmenty da się nazwać.
# TODO: dopasuj kmeans_model = KMeans(n_clusters=chosen_k, random_state=RANDOM_STATE, n_init=10) do X_cluster i zapisz etykiety w df_clean["segment"].

fig, ax = plt.subplots(figsize=(10, 6))
for segment_id in sorted(df_clean["segment"].unique()):
    part = df_clean[df_clean["segment"] == segment_id]
    ax.scatter(part["tenure"], part["MonthlyCharges"], s=12, alpha=0.6,
               color=SEGMENT_COLORS[segment_id % len(SEGMENT_COLORS)], label=f"Segment {segment_id}")
ax.set_title("Segmenty klientów: staż a opłata miesięczna")
ax.set_xlabel("Staż [miesiące]")
ax.set_ylabel("Opłata miesięczna")
ax.legend(markerscale=2)
plt.show()

In [ ]:
# Profil segmentów: średnie cech w oryginalnych jednostkach i liczba klientów
# TODO: policz średnie cluster_features w grupach df_clean["segment"] i dopisz kolumnę z liczbą klientów w segmencie.
segment_profile.round(1)

In [ ]:
# Dopiero teraz patrzymy na Churn: odsetek odejść w każdym segmencie
# TODO: policz średnią kolumny Churn w grupach według segment.
print((churn_by_segment * 100).round(1))

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar([f"Segment {s}" for s in churn_by_segment.index], churn_by_segment.values * 100,
       color=[SEGMENT_COLORS[s % len(SEGMENT_COLORS)] for s in churn_by_segment.index])
ax.set_title("Odsetek odejść w segmentach")
ax.set_xlabel("Segment")
ax.set_ylabel("Odsetek odejść [%]")
plt.show()

### Nazwy segmentów

Na podstawie tabeli profilu nadaj każdemu segmentowi krótką nazwę po polsku, na przykład według stażu i wysokości rachunku. Algorytm tych nazw nie zna, to Twoja interpretacja.

Segment 0: ...

Segment 1: ...

Segment 2: ...

In [ ]:
# Samokontrola: Zadanie 3
assert len(inertias) == 7 and len(silhouettes) == 7, "Listy inertias i silhouettes mają mieć po jednej wartości dla każdego k od 2 do 8."
assert all(-1 <= s <= 1 for s in silhouettes), "Silhouette musi leżeć między -1 a 1. Sprawdź, czy liczysz ją na X_cluster."
assert all(a > b for a, b in zip(inertias, inertias[1:])), "Inercja powinna maleć wraz z k. Sprawdź, czy dopisujesz inertia_ z modelu dla bieżącego k."
assert "chosen_k" in globals() and 2 <= chosen_k <= 8, "chosen_k ma być liczbą od 2 do 8."
assert "segment" in df_clean.columns, "Brakuje kolumny segment w df_clean."
assert df_clean["segment"].nunique() == chosen_k, "Liczba segmentów nie zgadza się z chosen_k."
assert df_clean["segment"].value_counts().min() >= 300, "Któryś segment ma mniej niż 300 klientów. Takiego segmentu firma nie obsłuży osobno, wybierz inne k."
assert "liczba_klientow" in segment_profile.columns, "W tabeli profilu brakuje kolumny liczba_klientow."
assert churn_by_segment.between(0, 1).all(), "Odsetki odejść w segmentach mają być ułamkami od 0 do 1."
print("Zadanie 3: OK")

## Zadanie 4: Klasyfikacja i ewaluacja (20 min)

Uzupełnij luki `# TODO`:

- podział na zbiór uczący i testowy: `train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)`,
- dokładność modelu naiwnego „nikt nie odchodzi" na zbiorze testowym w zmiennej `naive_accuracy`,
- `DecisionTreeClassifier(max_depth=4, random_state=42)` w zmiennej `tree_model`, dokładność na zbiorze uczącym i testowym,
- `classification_report` dla zbioru testowego. Odczytaj recall dla klasy 1,
- ten sam model z `class_weight="balanced"` w zmiennej `tree_balanced`,
- ważność cech `feature_importances_` jako tabela `importances`, posortowana malejąco.

Macierz pomyłek, tabela porównawcza, wykres ważności cech i rysunek drzewa są gotowe.

Pytanie kontrolne: co mówią pierwsze dwa podziały drzewa w języku biznesu?

In [ ]:
X = df_model.drop(columns="Churn")
y = df_model["Churn"]
# TODO: podziel X i y funkcją train_test_split z test_size=0.2, stratify=y i random_state=RANDOM_STATE.
print("Zbiór uczący:", X_train.shape, " zbiór testowy:", X_test.shape)
print(f"Odsetek odejść: uczący {y_train.mean():.1%}, testowy {y_test.mean():.1%}")

In [ ]:
# Model naiwny: przewidujemy, że nikt nie odchodzi
# TODO: policz, jaka część y_test to klasa 0, to dokładność modelu, który zawsze mówi 0.
print(f"Dokładność modelu naiwnego: {naive_accuracy:.3f}")

In [ ]:
# TODO: utwórz tree_model = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE), naucz go na zbiorze uczącym i policz train_accuracy oraz test_accuracy funkcją accuracy_score.
print(f"Drzewo: dokładność train {train_accuracy:.3f}, test {test_accuracy:.3f}")

In [ ]:
y_pred_plain = tree_model.predict(X_test)

fig, ax = plt.subplots(figsize=(10, 6))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_plain, display_labels=["zostaje", "odchodzi"],
                                        cmap=SEA_CMAP, colorbar=False, ax=ax)
ax.set_title("Macierz pomyłek: drzewo bez wag")
ax.set_xlabel("Przewidziana klasa")
ax.set_ylabel("Prawdziwa klasa")
plt.show()

# TODO: wypisz classification_report dla y_test i y_pred_plain z target_names i digits=3.

In [ ]:
# Ten sam model, ale błędy na klasie odejść ważą więcej
# TODO: utwórz tree_balanced tak jak tree_model, dodaj class_weight="balanced" i naucz go na zbiorze uczącym.
y_pred_balanced = tree_balanced.predict(X_test)

fig, ax = plt.subplots(figsize=(10, 6))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_balanced, display_labels=["zostaje", "odchodzi"],
                                        cmap=SEA_CMAP, colorbar=False, ax=ax)
ax.set_title("Macierz pomyłek: drzewo z class_weight=\"balanced\"")
ax.set_xlabel("Przewidziana klasa")
ax.set_ylabel("Prawdziwa klasa")
plt.show()

# Porównanie obu drzew na zbiorze testowym
metrics_table = pd.DataFrame({
    "bez wag": [
        accuracy_score(y_train, tree_model.predict(X_train)),
        accuracy_score(y_test, y_pred_plain),
        precision_score(y_test, y_pred_plain),
        recall_score(y_test, y_pred_plain),
        f1_score(y_test, y_pred_plain),
    ],
    "balanced": [
        accuracy_score(y_train, tree_balanced.predict(X_train)),
        accuracy_score(y_test, y_pred_balanced),
        precision_score(y_test, y_pred_balanced),
        recall_score(y_test, y_pred_balanced),
        f1_score(y_test, y_pred_balanced),
    ],
}, index=["dokładność train", "dokładność test", "precision (klasa 1)", "recall (klasa 1)", "F1 (klasa 1)"])
metrics_table.round(3)

In [ ]:
# Które cechy drzewo wykorzystuje najmocniej
# TODO: zbuduj pd.Series z tree_model.feature_importances_ z nazwami kolumn X i posortuj malejąco.
top10 = importances.head(10)
print(top10.round(3))

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top10.index[::-1], top10.values[::-1], color=SEA)
ax.set_title("10 najważniejszych cech drzewa")
ax.set_xlabel("Ważność cechy")
ax.set_ylabel("Cecha")
plt.tight_layout()
plt.show()

In [ ]:
# Dwa pierwsze poziomy drzewa do przeczytania na głos
fig, ax = plt.subplots(figsize=(20, 9))
plot_tree(tree_model, max_depth=2, feature_names=list(X.columns), class_names=["zostaje", "odchodzi"],
          filled=True, impurity=False, proportion=True, rounded=True, fontsize=14, ax=ax)
ax.set_title("Drzewo decyzyjne: dwa pierwsze poziomy")
plt.show()

print(export_text(tree_model, feature_names=list(X.columns), max_depth=2))

In [ ]:
# Samokontrola: Zadanie 4
assert "X_test" in globals() and len(X_test) == 1407, "Zbiór testowy ma mieć 20 % wierszy. Sprawdź test_size."
assert abs(y_test.mean() - y.mean()) < 0.002, "Odsetek odejść w zbiorze testowym odbiega od całości. Sprawdź parametr stratify."
assert abs(naive_accuracy - 0.735) < 0.001, "Dokładność modelu naiwnego się nie zgadza. Ten model zawsze przewiduje klasę 0."
assert tree_model.get_depth() <= 4, "Drzewo ma być płytkie: sprawdź max_depth."
assert test_accuracy > naive_accuracy, "Drzewo na teście nie bije modelu naiwnego. Sprawdź, na którym zbiorze uczysz, a na którym liczysz dokładność."
assert tree_balanced.class_weight == "balanced", "tree_balanced ma mieć class_weight=\"balanced\"."
assert recall_score(y_test, y_pred_balanced) > recall_score(y_test, y_pred_plain), "Recall klasy 1 w wersji balanced powinien być wyższy niż bez wag."
assert "importances" in globals() and len(importances) == X.shape[1], "Tabela importances ma mieć jedną wartość na każdą kolumnę X."
assert importances.is_monotonic_decreasing, "Posortuj importances malejąco."
print("Zadanie 4: OK")

## Zadanie 5: Decyzja biznesowa, próg i zysk (15 min)

Założenia:

- oferta utrzymaniowa kosztuje 50 zł na każdego klienta, którego model oflaguje,
- każdy naprawdę odchodzący klient, którego oflagujemy, zostaje i jest wart 400 zł. To uproszczenie: w praktyce oferta nie zatrzyma każdego.

Zysk = 400 × TP − 50 × (TP + FP), gdzie TP to oflagowani, którzy naprawdę odchodzą, a FP to oflagowani, którzy i tak by zostali.

Uzupełnij luki `# TODO`:

- prawdopodobieństwo odejścia z `predict_proba` dla obu drzew,
- funkcja `profit`, która liczy zysk dla danego progu,
- pętla po progach od 0,05 do 0,95 co 0,05 dla obu drzew,
- najlepszy próg i zysk dla każdego drzewa.

Wykres i tabela z porównaniem do progu 0,5 oraz do wariantów „dzwonimy do wszystkich" i „nie dzwonimy do nikogo" są gotowe.

Pytanie kontrolne: o ile złotych najlepszy próg drzewa bez wag bije domyślny próg 0,5? Na koniec zmień 400 na 150 i zobacz, jak przesuwa się najlepszy próg.

In [ ]:
OFFER_COST = 50        # koszt oferty na jednego oflagowanego klienta [zł]
CUSTOMER_VALUE = 400   # wartość zatrzymanego klienta [zł]

# Prawdopodobieństwo klasy 1 (odejście) dla każdego klienta ze zbioru testowego
# TODO: weź drugą kolumnę z tree_model.predict_proba(X_test).
# TODO: zrób to samo dla tree_balanced.
print("Przykładowe prawdopodobieństwa:", proba_plain[:5].round(3))

In [ ]:
def profit(y_true, proba, threshold):
    """Zysk kampanii: oflagowani to klienci z prawdopodobieństwem odejścia co najmniej równym progowi."""
    y_true = np.asarray(y_true)
    flagged = proba >= threshold
    # TODO: policz tp (oflagowani z y_true == 1) i fp (oflagowani z y_true == 0), potem zwróć zysk ze wzoru CUSTOMER_VALUE * tp - OFFER_COST * (tp + fp).


thresholds = np.round(np.arange(0.05, 0.96, 0.05), 2)
# TODO: dla każdego progu z thresholds policz zysk obu drzew funkcją profit i zapisz wyniki w listach profits_plain oraz profits_balanced.
profit_table = pd.DataFrame({"próg": thresholds, "zysk bez wag [zł]": profits_plain, "zysk balanced [zł]": profits_balanced})
profit_table

In [ ]:
# Warianty odniesienia: oferta dla wszystkich i dla nikogo
profit_everyone = profit(y_test, np.ones(len(y_test)), 0.5)
profit_nobody = 0

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(thresholds, profits_plain, marker="o", color=GOLD, linewidth=2, label="drzewo bez wag")
ax.plot(thresholds, profits_balanced, marker="o", color=SEA, linewidth=2, label="drzewo balanced")
ax.axhline(profit_everyone, color=GREY, linestyle="--", linewidth=2, label="oferta dla wszystkich")
ax.axhline(profit_nobody, color=GRAPHITE, linestyle=":", linewidth=2, label="oferta dla nikogo")
ax.axvline(0.5, color=GREY, linewidth=1)
ax.set_title("Zysk kampanii a próg decyzji")
ax.set_xlabel("Próg prawdopodobieństwa odejścia")
ax.set_ylabel("Zysk na zbiorze testowym [zł]")
ax.legend()
plt.show()

In [ ]:
# TODO: znajdź indeks największego zysku (np.argmax) w profits_plain i profits_balanced, a z niego najlepszy próg i zysk dla obu drzew.

profit_at_05_plain = profit(y_test, proba_plain, 0.5)
profit_at_05_balanced = profit(y_test, proba_balanced, 0.5)

summary = pd.DataFrame([
    ["drzewo bez wag, najlepszy próg", best_threshold_plain, best_profit_plain],
    ["drzewo bez wag, próg 0,5", 0.5, profit_at_05_plain],
    ["drzewo balanced, najlepszy próg", best_threshold_balanced, best_profit_balanced],
    ["drzewo balanced, próg 0,5", 0.5, profit_at_05_balanced],
    ["oferta dla wszystkich", None, profit_everyone],
    ["oferta dla nikogo", None, profit_nobody],
], columns=["wariant", "próg", "zysk [zł]"])
summary

In [ ]:
# Samokontrola: Zadanie 5
assert len(thresholds) == 19, "Progi mają iść od 0,05 do 0,95 co 0,05, czyli 19 wartości."
assert proba_plain.min() >= 0 and proba_plain.max() <= 1, "proba_plain ma zawierać prawdopodobieństwa od 0 do 1. Weź kolumnę klasy 1 z predict_proba."
assert profit(y_test, np.zeros(len(y_test)), 0.5) == 0, "Gdy nikogo nie flagujemy, zysk musi wynosić 0. Sprawdź funkcję profit."
assert len(profits_plain) == len(thresholds) and len(profits_balanced) == len(thresholds), "Listy zysków mają mieć po jednej wartości na każdy próg."
assert best_profit_plain == max(profits_plain), "best_profit_plain to nie jest największy zysk z listy."
assert best_profit_plain > profit_at_05_plain, "Najlepszy próg powinien dać więcej niż próg 0,5. Sprawdź pętlę po progach."
print("Zadanie 5: OK")

## Zadanie 6: dla szybkich (5 min)

Sprawdź las losowy `RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")` w tym samym rachunku zysku: ucz na zbiorze uczącym, policz `predict_proba` na testowym, zysk dla każdego progu i najlepszy próg.

Pytanie: czy mocniejszy model daje więcej złotówek niż dobrze dobrany próg prostego drzewa? Odpowiedz jednym zdaniem w komórce niżej.

In [ ]:
# TODO: naucz rf_model = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, class_weight="balanced"), policz proba_rf i listę profits_rf dla thresholds.
best_idx_rf = int(np.argmax(profits_rf))
best_threshold_rf, best_profit_rf = thresholds[best_idx_rf], profits_rf[best_idx_rf]
print(f"Las losowy: dokładność test {accuracy_score(y_test, rf_model.predict(X_test)):.3f}, "
      f"recall klasy 1 {recall_score(y_test, rf_model.predict(X_test)):.3f}")
print(f"Las losowy: najlepszy próg {best_threshold_rf}, zysk {best_profit_rf} zł, zysk przy progu 0,5: {profit(y_test, proba_rf, 0.5)} zł")
print(f"Drzewo bez wag: najlepszy próg {best_threshold_plain}, zysk {best_profit_plain} zł")

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(thresholds, profits_plain, marker="o", color=GOLD, linewidth=2, label="drzewo bez wag")
ax.plot(thresholds, profits_balanced, marker="o", color=SEA, linewidth=2, label="drzewo balanced")
ax.plot(thresholds, profits_rf, marker="o", color=PLUM, linewidth=2, label="las losowy balanced")
ax.axhline(profit_everyone, color=GREY, linestyle="--", linewidth=2, label="oferta dla wszystkich")
ax.set_title("Zysk kampanii a próg: drzewa i las losowy")
ax.set_xlabel("Próg prawdopodobieństwa odejścia")
ax.set_ylabel("Zysk na zbiorze testowym [zł]")
ax.legend()
plt.show()

### Twoja odpowiedź

Czy las losowy daje więcej złotówek niż drzewo z dobrze dobranym progiem? Jedno zdanie.

Odpowiedź: ...

In [ ]:
# Samokontrola: Zadanie 6
assert "rf_model" in globals() and rf_model.n_estimators == 200, "rf_model ma mieć 200 drzew."
assert rf_model.class_weight == "balanced", "rf_model ma mieć class_weight=\"balanced\"."
assert len(profits_rf) == len(thresholds), "Lista profits_rf ma mieć po jednej wartości na każdy próg."
print("Zadanie 6: OK")

## Zakończenie

1. Liczy się proces, a nie wybór algorytmu: od surowego CSV do decyzji, ile wydać na kampanię.
2. Przygotowanie danych to decyzje. Zanim usuniesz wiersze z brakami, sprawdź, kim są.
3. Metryka ma wynikać z kosztu. Próg 0,5 jest przypadkowy, firma liczy złotówki.

Rozwiązanie trafi do repozytorium po zajęciach.

Pytanie do sali: co byście sprawdzili, zanim ten model trafi na produkcję?

## Ściąga: pandas i scikit-learn

Każdy wpis: co robi, potem przykład. Nazwy zmiennych jak w notatniku.

Wczytanie CSV z adresu URL albo pliku.
`df = pd.read_csv(DATA_URL)`

Rozmiar tabeli (wiersze, kolumny) i pierwsze wiersze.
`df.shape; df.head()`

Udział każdej wartości w kolumnie (ułamki zamiast liczebności).
`df["Contract"].value_counts(normalize=True)`

Średnia w grupach: odsetek odejść w każdym rodzaju umowy.
`(df["Churn"] == "Yes").groupby(df["Contract"]).mean()`

Średnie kilku kolumn w grupach.
`df_clean.groupby("segment")[["tenure", "MonthlyCharges"]].mean()`

Tekst na liczby; czego nie da się zamienić, staje się NaN.
`df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce")`

Wiersze z brakiem w kolumnie.
`df_clean[df_clean["TotalCharges"].isna()]`

Usunięcie wierszy z brakiem w wybranej kolumnie.
`df_clean = df_clean.dropna(subset=["TotalCharges"])`

Usunięcie kolumny.
`df_clean = df_clean.drop(columns="customerID")`

Zamiana wartości według słownika.
`df_clean["Churn"] = df_clean["Churn"].map({"Yes": 1, "No": 0})`

Kolumny kategoryczne na kolumny 0/1, bez pierwszej kategorii.
`df_model = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True, dtype=int)`

Standaryzacja: średnia 0, odchylenie 1 w każdej kolumnie.
`X_cluster = StandardScaler().fit_transform(df_clean[["tenure", "MonthlyCharges", "TotalCharges"]])`

K-means z ustalonym ziarnem; `fit_predict` zwraca numer segmentu dla każdego wiersza.
`kmeans_model = KMeans(n_clusters=3, random_state=42, n_init=10); labels = kmeans_model.fit_predict(X_cluster)`

Inercja: suma kwadratów odległości od środków segmentów (mniej = ciaśniej).
`kmeans_model.inertia_`

Silhouette: od -1 do 1, im wyżej, tym lepiej rozdzielone segmenty.
`silhouette_score(X_cluster, labels)`

Podział na zbiór uczący i testowy z zachowaniem proporcji klas.
`X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)`

Drzewo decyzyjne: utworzenie i nauka.
`tree_model = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train, y_train)`

Drzewo, w którym błędy na rzadszej klasie ważą więcej.
`DecisionTreeClassifier(max_depth=4, random_state=42, class_weight="balanced")`

Dokładność na zbiorze testowym.
`accuracy_score(y_test, tree_model.predict(X_test))`

Macierz pomyłek jako wykres.
`ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["zostaje", "odchodzi"])`

Precision, recall i F1 dla każdej klasy.
`print(classification_report(y_test, y_pred, digits=3))`

Prawdopodobieństwo klasy 1 (odejście) dla każdego klienta.
`proba = tree_model.predict_proba(X_test)[:, 1]`

Decyzja według własnego progu zamiast 0,5.
`flagged = proba >= 0.3`

Ważność cech, posortowana malejąco.
`pd.Series(tree_model.feature_importances_, index=X.columns).sort_values(ascending=False)`

Rysunek dwóch pierwszych poziomów drzewa.
`plot_tree(tree_model, max_depth=2, feature_names=list(X.columns), filled=True, fontsize=14)`